# Chirurgie à chaud EN COURS d'un vrai entraînement + analyse causale de la couche greffée

Fusionne trois expériences du plan de session (P1+P2+P3) en une seule, cohérente :

1. **Phase A** : entraîner le char-LM (TinyShakespeare, dim=256/4 têtes/4 couches,
   `batched_attn=true`) pendant 10 000 pas -- la moitié du run de référence de
   `real_llm.ipynb` (déjà validé : val loss finale 1.6229 à 20 000 pas, Δ0.0033
   nats avec le miroir PyTorch).
2. **Baseline** : recherche de circuit d'induction sur texte réel (noms de
   personnages répétés dans TinyShakespeare, ex. "MENENIUS:") sur le modèle à
   4 couches, à mi-parcours -- `greedy_patch_search!`/`backward_prune!` avec le
   nouveau kwarg `metric` (recovery restreinte à une ligne, comme `induction.ipynb`
   mais sur du texte réel plutôt que sur une tâche synthétique).
3. **Chirurgie** : `insert_block!` (avec le nouveau kwarg `batched_attn=true` pour
   rester homogène) insère une 5ème couche EN COURS d'entraînement -- preuve F1
   (texte généré identique bit-à-bit juste avant/après insertion).
4. **Phase B** : poursuite de l'entraînement (5 couches) pour les 10 000 pas
   restants, moments AdamW fusionnés par nom de paramètre (patron F4 de
   `test/test_surgery.jl`), pas de réinitialisation du pas `t` global.
5. **Final** : même recherche de circuit sur les mêmes fenêtres gelées, maintenant
   sur 5 couches -- la nouvelle couche a-t-elle acquis une responsabilité causale
   mesurable ? Verdict honnête contre des critères falsifiables, quel que soit
   le résultat.

Aucune modification de `real_llm.ipynb` (artefact de parité déjà validé, laissé
intact). Deux correctifs `src/` de cette session sont des prérequis directs :
`greedy_patch_search!`/`backward_prune!` ne perdent plus le patch d'un site déjà
retenu quand un candidat amont est testé (bug trouvé et corrigé le 2026-07-10),
et `insert_block!`/les deux fonctions de recherche acceptent maintenant
`batched_attn`/`metric`.

In [1]:
using NeuroDSL, Random, Statistics, Printf, StatsPlots

dev = NeuroDSL.Backend.CUDADevice()
ns = :real_llm_surgery
println("Device: ", dev)

Device: NeuroDSL.Backend.CUDADevice()


## 1. Corpus : TinyShakespeare (téléchargé une seule fois, hors ligne ensuite)

In [2]:
using Downloads

const CORPUS_URL  = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
const CORPUS_PATH = joinpath(@__DIR__, "data", "tinyshakespeare", "input.txt")

function load_corpus(path::String, url::String)
    if !isfile(path)
        mkpath(dirname(path))
        Downloads.download(url, path)
    end
    text = read(path, String)
    println("Corpus chargé : ", length(text), " caractères depuis ", path)
    return text
end

text = load_corpus(CORPUS_PATH, CORPUS_URL)
println(first(text, 200))

Corpus chargé : 1115394 caractères depuis C:\Users\Nevermind\Desktop\NeuroDSL\notebook\data\tinyshakespeare\input.txt
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


## 2. Tokenizer caractère (vrai texte -> vrais IDs, pas de BPE, zéro dépendance nouvelle)

In [3]:
function build_char_tokenizer(text::String)
    chars = sort(unique(collect(text)))
    stoi = Dict(c => i for (i, c) in enumerate(chars))
    return chars, stoi
end

encode(text::AbstractString, stoi::Dict{Char,Int}) = [stoi[c] for c in text]
decode(ids::AbstractVector{<:Integer}, chars::Vector{Char}) = String(chars[ids])

chars, stoi = build_char_tokenizer(text)
vocab_size = length(chars)
println("vocab_size = ", vocab_size)
println("10 premiers caractères (triés) = ", chars[1:10])
println("(à comparer avec les 10 premiers caractères imprimés par real_llm_py.py -- doivent être identiques)")

vocab_size = 65
10 premiers caractères (triés) = ['\n', ' ', '!', '$', '&', '\'', ',', '-', '.', '3']
(à comparer avec les 10 premiers caractères imprimés par real_llm_py.py -- doivent être identiques)


## 3. Split train/validation (90/10 par position -- le val est la FIN du corpus, jamais vue à l'entraînement)

In [4]:
data = encode(text, stoi)
n_total = length(data)
n_train = floor(Int, 0.9 * n_total)
train_ids = data[1:n_train]
val_ids   = data[n_train+1:end]
println("train: ", length(train_ids), " caractères  |  val: ", length(val_ids), " caractères")

train: 1003854 caractères  |  val: 111540 caractères


## 4. Construction du graphe (copie directe de `build_induction_graph`, généralisée à un vrai vocabulaire/contexte)

In [5]:
function build_char_lm_graph(dev, ns::Symbol; vocab_size::Int, dim::Int, n_heads::Int,
                              hidden_dim::Int, n_layers::Int, block_size::Int, batched::Bool=true)
    g = NeuroDSL.NeuroGraph(namespace=ns, device=dev)
    NeuroDSL.set!(g, :token_ids, ones(Int, block_size); atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.set!(g, :pos_ids, collect(1:block_size); atom_type=NeuroDSL.Datom, namespace=ns)
    tok_emb = NeuroDSL.Embedding(vocab_size, dim)(g, :token_ids, :tok; namespace=ns)
    pos_emb = NeuroDSL.Embedding(block_size, dim)(g, :pos_ids, :pos; namespace=ns)
    x = :embed_sum
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(x, [tok_emb, pos_emb], :add; namespace=ns))
    # `batched` (défaut true) exposé en kwarg pour P1-bis (graphe jumeau non-batché,
    # comparaison de coût/cône de patch) -- src/layers.jl, conçu avec Fable.
    out = NeuroDSL.LlamaModel(n_layers, dim, n_heads, hidden_dim; batched_attn=batched)(g, x; namespace=ns)
    logits = NeuroDSL.Linear(dim, vocab_size)(g, out, :lm_head; namespace=ns)
    NeuroDSL.set!(g, :labels, ones(Int, block_size); atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:loss, [logits, :labels], :cross_entropy; namespace=ns))
    return g, logits
end

# ── Hyperparamètres (voir plan : dimensionnés pour rester bien sous le max GPU
# déjà confirmé cette session -- dim=1024 en forward+backward réel) ──────────
block_size = 256
dim        = 256
n_heads    = 4
hidden_dim = 512
n_layers   = 4

g, logits_sym = build_char_lm_graph(dev, ns; vocab_size=vocab_size, dim=dim, n_heads=n_heads,
                                     hidden_dim=hidden_dim, n_layers=n_layers, block_size=block_size)
ps = NeuroDSL.params(g; namespace=ns)
n_scalars = sum(length(p.value) for p in ps)
println("Graphe : ", length(g.nodes[ns]), " nœuds, ", length(ps), " tenseurs de paramètres, ",
        n_scalars, " scalaires (~", round(n_scalars/1e6, digits=2), "M)")

Graphe : 220 nœuds, 40 tenseurs de paramètres, 2722369 scalaires (~2.72M)


## 5. Échantillonnage de fenêtres réelles + perte de validation + génération autorégressive

In [6]:
function sample_window(rng, ids::Vector{Int}, block_size::Int)
    i = rand(rng, 1:(length(ids) - block_size))
    tokens = ids[i:i+block_size-1]
    labels = ids[i+1:i+block_size]
    return tokens, labels
end

# 64 fenêtres FIXES également espacées dans le split val -- déterministe,
# comparable entre checkpoints. Jamais de backward_graph! ici (pas de fuite
# du val dans les gradients) -- donc `demand_release!` (src/demand_release.jl)
# est sûr : libère les activations intermédiaires au fil du calcul au lieu de
# les garder résidentes jusqu'au prochain train_char_lm! (mêmes résultats,
# vérifié bit-à-bit cette session -- réduit juste le pic VRAM de cet appel).
function val_loss(g::NeuroDSL.NeuroGraph, ns::Symbol; val_ids::Vector{Int}, block_size::Int, n_windows::Int=64)
    max_start = length(val_ids) - block_size
    starts = round.(Int, range(1, max_start, length=n_windows))
    total = 0.0
    for i in starts
        tokens = val_ids[i:i+block_size-1]
        labels = val_ids[i+1:i+block_size]
        NeuroDSL.set!(g, :token_ids, tokens; atom_type=NeuroDSL.Datom, namespace=ns)
        NeuroDSL.set!(g, :pos_ids, collect(1:block_size); atom_type=NeuroDSL.Datom, namespace=ns)
        NeuroDSL.set!(g, :labels, labels; atom_type=NeuroDSL.Datom, namespace=ns)
        NeuroDSL.invalidate_all!(g; namespace=ns)
        loss_val = NeuroDSL.demand_release!(g, :loss; namespace=ns)
        total += Float64(sum(Array(loss_val)))
    end
    return total / n_windows
end

# Génération autorégressive -- échantillonnage AVEC TEMPÉRATURE (pas argmax) :
# l'argmax sur un char-LM dégénère quasi systématiquement en boucles
# répétitives ("the the the..."), donnant une fausse impression d'échec alors
# que la distribution apprise est bonne. C'est ce que nanoGPT/char-rnn font
# pour leurs démos.
function generate_text(g::NeuroDSL.NeuroGraph, logits_sym::Symbol, ns::Symbol,
                        stoi::Dict{Char,Int}, chars::Vector{Char};
                        seed_text::String="\n", n_chars::Int=300, temperature::Float32=0.8f0,
                        block_size::Int, rng=MersenneTwister(777), use_argmax::Bool=false)
    ctx = encode(seed_text, stoi)
    generated = Char[]
    for _ in 1:n_chars
        window = length(ctx) > block_size ? ctx[end-block_size+1:end] : ctx
        t = length(window)
        NeuroDSL.set!(g, :token_ids, window; atom_type=NeuroDSL.Datom, namespace=ns)
        NeuroDSL.set!(g, :pos_ids, collect(1:t); atom_type=NeuroDSL.Datom, namespace=ns)
        NeuroDSL.invalidate_all!(g; namespace=ns)
        row = Array(NeuroDSL.demand_release!(g, logits_sym; namespace=ns))[end, :]
        local next_id
        if use_argmax
            next_id = argmax(row)
        else
            p = exp.((row .- maximum(row)) ./ temperature)
            p ./= sum(p)
            r = rand(rng)
            cum = 0.0f0
            next_id = length(p)
            for (idx, pi) in enumerate(p)
                cum += pi
                if r <= cum
                    next_id = idx
                    break
                end
            end
        end
        push!(ctx, next_id)
        push!(generated, chars[next_id])
    end
    return String(generated)
end

generate_text (generic function with 1 method)

## 6. Vérification de sanité : perte initiale ≈ ln(vocab_size)

In [7]:
rng_check = MersenneTwister(1)
tokens0, labels0 = sample_window(rng_check, train_ids, block_size)
NeuroDSL.set!(g, :token_ids, tokens0; atom_type=NeuroDSL.Datom, namespace=ns)
NeuroDSL.set!(g, :pos_ids, collect(1:block_size); atom_type=NeuroDSL.Datom, namespace=ns)
NeuroDSL.set!(g, :labels, labels0; atom_type=NeuroDSL.Datom, namespace=ns)
NeuroDSL.invalidate_all!(g; namespace=ns)
loss0 = Float64(sum(Array(NeuroDSL.demand!(g, :loss; namespace=ns))))
@printf("Perte initiale mesurée : %.4f   (attendu ln(%d) = %.4f)\n", loss0, vocab_size, log(vocab_size))
@assert abs(loss0 - log(vocab_size)) < 0.5 "Perte initiale trop loin de ln(vocab_size) -- vérifier le câblage avant d'entraîner"

println("\n--- Échantillon AVANT tout entraînement (poids aléatoires) ---")
sample_before = generate_text(g, logits_sym, ns, stoi, chars; block_size=block_size, rng=MersenneTwister(777))
println(sample_before)

Perte initiale mesurée : 4.2096   (attendu ln(65) = 4.1744)

--- Échantillon AVANT tout entraînement (poids aléatoires) ---
3;qpoj
IXKfj&CWwfPgvBQ!L-xqGBJ!vIF!xe'!IsGE$Z&!'
qUftj.jnoaVA?!uiY'xxIWiOfXxKn;b3orU
XpMQL
xZg:Y
mqw:LTdhvlRo
$xx
&oNNCCisG3sJ3WJn&ZS-mwjhPWK3cwCVb?LnqbhrLsaTeyh
i&W Eg

FgWS
3j&NHps!vfH.nP,yJUo?UuorBAX3N,vIqrBYJcT
wax-!EbxwABJmSvVo?ww-3vnwOl.zP!tjj!,b&zI
;KzYAYIEIW?NjmR.ojl
jRZcR.WA l'ig&pI.JZgox!u


## 7. Budget de calcul (estimé par Fable, à partir du chronométrage réel de `real_llm.ipynb`)

Référence mesurée : 505.7 s / 20 000 pas = 25.3 ms/pas à 4 couches (GPU RTX A5500).
Estimation pour cette expérience : Phase A (10k pas, 4 couches) ≈ 253 s ; Phase B
(10k pas, 5 couches) ≈ 316 s (borne sup à 31.6 ms/pas) ; les deux recherches de
circuit (gloutonne + élagage, 3 fenêtres, 16 puis 20 candidats) ≈ moins d'une
minute au total (chaque mesure est un `demand!` incrémental + une restauration
par copie de cache, pas un forward complet). **Total estimé : ~12-16 minutes.**
Le coût est dominé par l'entraînement, pas par l'interprétabilité -- c'est un
résultat en soi, cohérent avec la thèse du framework.

## 8. Phase A : 10 000 premiers pas (4 couches)

`train_char_lm!` v2 : accepte maintenant `rng` (objet, pas une graine entière),
`t0` (pas de départ, pour que le compteur AdamW `t` ne se réinitialise jamais à
la frontière de la greffe) et `moments` (`Dict{Symbol,Tuple}` keyé par NOM de
paramètre, patron F4 de `test/test_surgery.jl` -- après `insert_block!`,
`params(g)` peut réordonner, donc réutiliser des `Vector`s positionnels d'une
phase à l'autre serait incorrect). Retourne `moments` et `rng` en plus des
métriques habituelles, pour les transmettre tels quels à la Phase B.

In [8]:
function train_char_lm!(g::NeuroDSL.NeuroGraph, ns::Symbol, logits_sym::Symbol,
                         stoi::Dict{Char,Int}, chars::Vector{Char};
                         train_ids::Vector{Int}, val_ids::Vector{Int}, block_size::Int,
                         n_steps::Int, lr::Float32=1f-3, b1::Float32=0.9f0, b2::Float32=0.999f0,
                         eps_v::Float32=1f-8, clip::Float32=1f0, wd::Float32=0f0,
                         rng::MersenneTwister=MersenneTwister(123), t0::Int=0,
                         moments::Union{Nothing,Dict{Symbol,Tuple{Any,Any}}}=nothing,
                         val_every::Int=500, sample_steps=(), norm_watch::Vector{Symbol}=Symbol[])
    dev = g.device
    ps = NeuroDSL.params(g; namespace=ns)
    # Moments keyés par NOM (pas par position) -- patron F4 (test/test_surgery.jl) :
    # un paramètre déjà connu (avant une éventuelle greffe) reprend ses moments
    # exacts ; un paramètre nouveau (apporté par insert_block!) démarre à zéro.
    m1s = Vector{Any}(undef, length(ps))
    m2s = Vector{Any}(undef, length(ps))
    for (i, p) in enumerate(ps)
        if moments !== nothing && haskey(moments, p.name)
            m1s[i], m2s[i] = moments[p.name]
        else
            m1s[i] = NeuroDSL.Backend.zeros32(dev, size(p.value)...)
            m2s[i] = NeuroDSL.Backend.zeros32(dev, size(p.value)...)
        end
    end
    train_losses = Float64[]
    val_history = Tuple{Int,Float64}[]
    norm_history = NamedTuple[]   # (; step, norms::Dict{Symbol,Float64}) -- vide si norm_watch vide

    t_start = time()
    for t in (t0+1):(t0+n_steps)
        tokens, labels = sample_window(rng, train_ids, block_size)
        NeuroDSL.set!(g, :token_ids, tokens; atom_type=NeuroDSL.Datom, namespace=ns)
        NeuroDSL.set!(g, :pos_ids, collect(1:block_size); atom_type=NeuroDSL.Datom, namespace=ns)
        NeuroDSL.set!(g, :labels, labels; atom_type=NeuroDSL.Datom, namespace=ns)
        NeuroDSL.invalidate_all!(g; namespace=ns)
        loss_val = NeuroDSL.demand!(g, :loss; namespace=ns)
        push!(train_losses, Float64(sum(Array(loss_val))))
        NeuroDSL.backward_graph!(g, :loss; namespace=ns)
        NeuroDSL.adamw_step_batched!(dev, [p.value for p in ps], [p.gradient for p in ps],
                                     m1s, m2s, lr, b1, b2, eps_v, t, clip, wd)
        NeuroDSL.invalidate_all!(g; namespace=ns)

        if t % val_every == 0 || t == t0 + 1
            vl = val_loss(g, ns; val_ids=val_ids, block_size=block_size)
            push!(val_history, (t, vl))
            @printf("step %6d | train %.4f | val %.4f | ppl %.2f | bits/char %.3f\n",
                    t, train_losses[end], vl, exp(vl), vl/log(2))
            if !isempty(norm_watch)
                norms = Dict{Symbol,Float64}(s => norm(Array(NeuroDSL.node(g, s; namespace=ns).value)) for s in norm_watch)
                push!(norm_history, (; step=t, norms))
            end
        end
        if t in sample_steps
            s = generate_text(g, logits_sym, ns, stoi, chars; block_size=block_size, rng=MersenneTwister(777))
            println("\n--- Échantillon @ pas $t ---\n", s, "\n")
        end
    end
    elapsed = time() - t_start
    final_moments = Dict{Symbol,Tuple{Any,Any}}(p.name => (m1s[i], m2s[i]) for (i, p) in enumerate(ps))

    return (; train_losses, val_history, elapsed, moments=final_moments, rng, norm_history)
end

rng_A = MersenneTwister(123)
n_steps_A = 10_000
result_A = train_char_lm!(g, ns, logits_sym, stoi, chars;
                           train_ids=train_ids, val_ids=val_ids, block_size=block_size,
                           n_steps=n_steps_A, lr=1f-3, rng=rng_A, t0=0)
@printf("\nPhase A terminée : %d pas en %.1f s (%.2f ms/pas moyen)\n",
        n_steps_A, result_A.elapsed, 1000*result_A.elapsed/n_steps_A)

step      1 | train 4.1906 | val 3.7366 | ppl 41.95 | bits/char 5.391
step    500 | train 2.5849 | val 2.5502 | ppl 12.81 | bits/char 3.679
step   1000 | train 2.4285 | val 2.5478 | ppl 12.78 | bits/char 3.676
step   1500 | train 2.4821 | val 2.5275 | ppl 12.52 | bits/char 3.646
step   2000 | train 2.4118 | val 2.5129 | ppl 12.34 | bits/char 3.625
step   2500 | train 2.3230 | val 2.3928 | ppl 10.94 | bits/char 3.452
step   3000 | train 2.0782 | val 2.2866 | ppl 9.84 | bits/char 3.299
step   3500 | train 2.1500 | val 2.1863 | ppl 8.90 | bits/char 3.154
step   4000 | train 2.1176 | val 2.1494 | ppl 8.58 | bits/char 3.101
step   4500 | train 1.8423 | val 2.0499 | ppl 7.77 | bits/char 2.957
step   5000 | train 1.9304 | val 1.9817 | ppl 7.25 | bits/char 2.859
step   5500 | train 1.9230 | val 1.9654 | ppl 7.14 | bits/char 2.835
step   6000 | train 1.6730 | val 1.9373 | ppl 6.94 | bits/char 2.795
step   6500 | train 1.7565 | val 1.9165 | ppl 6.80 | bits/char 2.765
step   7000 | train 1.7225 |

## 9. Sélection et gel de 6 fenêtres de test (plafond-aware, diverses)

**v2 -- corrige un défaut trouvé après coup dans la v1** : sélectionner par
plus grande taille d'effet seule choisit mécaniquement les cas où le circuit
4-couches est déjà quasi-saturé (recovery 0.95-1.0), laissant zéro marge pour
qu'une couche greffée puisse jamais y contribuer. v2 ajoute un **indice de
plafond** (`ceiling`) : la recovery obtenue en patchant d'un coup les 4 têtes
de couche 1 -- si ce plafond est déjà proche de 1, il ne reste structurellement
rien à expliquer sur cette fenêtre. Sélection : **une seule fenêtre par nom de
personnage** (règle dure -- en v1, les 3 fenêtres avaient toutes atterri sur
"PETRUCHIO"), triées par plafond croissant (les moins saturées en premier), au
nombre de **6** (au lieu de 3). Une **variante "long_gap"** (3 en-têtes A→B→A,
nom différent entre les deux occurrences cibles -- copie à plus longue portée
avec distracteur) est aussi détectée et taguée, potentiellement moins couverte
par des têtes à portée courte.

In [9]:
using LinearAlgebra

# ── Énumération de TOUS les en-têtes de locuteur du split val, position absolue. ──
function all_speaker_headers(val_ids::Vector{Int}, chars::Vector{Char}; min_name_len::Int=4)
    text_val = decode(val_ids, chars)
    headers = NamedTuple[]
    for m in eachmatch(r"\n([A-Z][A-Z ]{2,})\:", text_val)
        name = String(strip(m.captures[1]))
        length(name) >= min_name_len || continue
        push!(headers, (; pos=m.offset + 1, name))
    end
    return headers
end

headers = all_speaker_headers(val_ids, chars)
println("En-têtes de locuteur trouvés dans le split val : ", length(headers))

# ── Candidats : paires de même nom tenant dans une fenêtre de block_size chars.
# "long_gap" si un en-tête de nom DIFFÉRENT s'intercale entre les deux (A→B→A).
function build_candidates(headers, block_size::Int; k::Int=3, margin::Int=5)
    candidates = NamedTuple[]
    n = length(headers)
    for i in 1:n-1
        for jx in i+1:n
            gap = headers[jx].pos - headers[i].pos
            gap > block_size - 20 && break   # headers triés par position -- au-delà, ça ne peut qu'empirer
            headers[i].name != headers[jx].name && continue
            kk = min(k, length(headers[i].name) - 1)
            kk < 1 && continue
            window_start = headers[i].pos - margin
            window_start < 1 && continue
            p1 = headers[i].pos - window_start + 1
            p2 = headers[jx].pos - window_start + 1
            p2 + kk > block_size && continue
            has_intervening_different = any(headers[m2].pos > headers[i].pos && headers[m2].pos < headers[jx].pos &&
                                             headers[m2].name != headers[i].name for m2 in i+1:jx-1)
            variant = has_intervening_different ? :long_gap : :adjacent
            push!(candidates, (; window_start, p1, p2, k=kk, name=headers[i].name, gap, variant))
        end
    end
    return candidates
end

candidates_raw = build_candidates(headers, block_size)
println("Candidats bruts (paires de même nom dans une fenêtre) : ", length(candidates_raw),
        "  (dont long_gap : ", count(c -> c.variant == :long_gap, candidates_raw), ")")
@assert length(candidates_raw) >= 6 "Pas assez de candidats -- revoir le protocole avant de continuer"

# Taille d'effet + indice de PLAFOND (recovery en patchant d'un coup les 4
# têtes de couche 1 -- mesure si le circuit 4-couches sature déjà cette fenêtre),
# tous les deux mesurés sur le modèle DÉJÀ entraîné (Phase A).
function effect_and_ceiling(g, ns, logits_sym, tokens_clean, tokens_corrupt, block_size, j, n_heads::Int)
    NeuroDSL.set!(g, :token_ids, tokens_clean; atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.set!(g, :pos_ids, collect(1:block_size); atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.invalidate_all!(g; namespace=ns)
    clean_logits = copy(Array(NeuroDSL.demand!(g, logits_sym; namespace=ns)))
    clean_cache  = NeuroDSL.capture_activations(g, ns)

    NeuroDSL.set!(g, :token_ids, tokens_corrupt; atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.invalidate_all!(g; namespace=ns)
    corrupt_logits = copy(Array(NeuroDSL.demand!(g, logits_sym; namespace=ns)))

    effect = norm(clean_logits[j, :] .- corrupt_logits[j, :])

    layer1_heads = [Symbol(:layer_1_mha_ao_h, h) for h in 1:n_heads]
    NeuroDSL.patch_nodes!(g, layer1_heads, clean_cache; namespace=ns)
    patched_logits = Array(NeuroDSL.demand!(g, logits_sym; namespace=ns))
    ceiling = NeuroDSL.recovery_metric(patched_logits[j:j, :], clean_logits[j:j, :], corrupt_logits[j:j, :])

    NeuroDSL.set!(g, :token_ids, tokens_corrupt; atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.invalidate_all!(g; namespace=ns)
    NeuroDSL.demand!(g, logits_sym; namespace=ns)
    return effect, ceiling
end

scored = NamedTuple[]
rng_scan = MersenneTwister(999)
for c in candidates_raw
    tokens_clean = val_ids[c.window_start:c.window_start+block_size-1]
    j = c.p2 + c.k - 1
    (j < 1 || j > block_size) && continue
    tokens_corrupt = copy(tokens_clean)
    orig_id = tokens_corrupt[c.p1 + c.k]
    new_id = orig_id
    while new_id == orig_id
        new_id = rand(rng_scan, 1:vocab_size)
    end
    tokens_corrupt[c.p1 + c.k] = new_id
    effect, ceiling = effect_and_ceiling(g, ns, logits_sym, tokens_clean, tokens_corrupt, block_size, j, n_heads)
    push!(scored, (; c..., j, tokens_clean, tokens_corrupt, effect, ceiling))
end
println("Candidats scorés (effet + plafond) : ", length(scored))

effect_max = maximum(x -> x.effect, scored)
effect_floor = max(1.0, 0.25 * effect_max)
eligible = filter(x -> x.effect >= effect_floor, scored)
println("Candidats au-dessus du seuil d'effet (", round(effect_floor, digits=3), ") : ", length(eligible))

# Un seul candidat par nom -- le MOINS saturé (plafond le plus bas) parmi ceux
# au-dessus du seuil d'effet. Corrige directement le défaut "3x le même nom" de v1.
best_per_name = Dict{String,eltype(eligible)}()
for x in eligible
    if !haskey(best_per_name, x.name) || x.ceiling < best_per_name[x.name].ceiling
        best_per_name[x.name] = x
    end
end
by_ceiling = sort(collect(values(best_per_name)), by = x -> x.ceiling)
println("Noms de personnages distincts disponibles : ", length(by_ceiling))

K = min(6, length(by_ceiling))
frozen_windows = by_ceiling[1:K]
n_long_gap = count(x -> x.variant == :long_gap, frozen_windows)

println("\nFenêtres GELÉES (", K, ", triées par plafond croissant -- une seule par nom) :")
for fw in frozen_windows
    @printf("  nom=%-14s variant=%-9s gap=%-4d k=%d  j=%-4d  effet=%.3f  plafond=%.4f\n",
            fw.name, fw.variant, fw.gap, fw.k, fw.j, fw.effect, fw.ceiling)
end
println("Fenêtres 'long_gap' parmi les gelées : ", n_long_gap, "/", K)
if minimum(x -> x.ceiling, frozen_windows) > 0.7
    println("\n⚠️  Aucune fenêtre gelée n'est sous le plafond de 0.7 -- l'induction char-level semble")
    println("   déjà largement saturée par la couche 1 dès ", n_steps_A, " pas. On gèle quand même les")
    println("   6 fenêtres les MOINS saturées disponibles, et on rapporte cette limite honnêtement.")
end

En-têtes de locuteur trouvés dans le split val : 882
Candidats bruts (paires de même nom dans une fenêtre) : 512  (dont long_gap : 482)
Candidats scorés (effet + plafond) : 512
Candidats au-dessus du seuil d'effet (2.503) : 8
Noms de personnages distincts disponibles : 6

Fenêtres GELÉES (6, triées par plafond croissant -- une seule par nom) :
  nom=GRUMIO         variant=long_gap  gap=40   k=3  j=48    effet=3.231  plafond=0.8405
  nom=ANTONIO        variant=long_gap  gap=53   k=3  j=61    effet=3.563  plafond=0.9772
  nom=PROSPERO       variant=long_gap  gap=76   k=3  j=84    effet=4.111  plafond=0.9922
  nom=GREMIO         variant=long_gap  gap=117  k=3  j=125   effet=3.895  plafond=0.9961
  nom=BIANCA         variant=long_gap  gap=88   k=3  j=96    effet=3.119  plafond=0.9972
  nom=ARIEL          variant=long_gap  gap=64   k=3  j=72    effet=10.012  plafond=0.9982
Fenêtres 'long_gap' parmi les gelées : 6/6

⚠️  Aucune fenêtre gelée n'est sous le plafond de 0.7 -- l'induction char-l

In [10]:
"""
    find_circuit!(g, ns, logits_sym, window; max_sites=6)

Recherche de circuit sur une fenêtre gelée : capture les caches propre/corrompu
LOCALEMENT (variables locales, jamais de globale réutilisée entre appels --
aucune fuite possible d'un cache périmé après une mutation de graphe entre deux
appels), candidats = toutes les sorties de tête `*_mha_ao_h{h}` du graphe
COURANT (16 avant greffe, 20 après), métrique = recovery restreinte à la ligne
`window.j` (via le nouveau kwarg `metric`, sinon la sortie entière noierait un
effet localisé à une seule position dans une séquence de 256 caractères).
`greedy_patch_search!`/`backward_prune!` réappliquent maintenant `selected` à
chaque mutation (correctif du 2026-07-10) -- sûr même si un site tardif est
retenu avant qu'un site précoce ne soit testé, exactement le cas d'un vrai
circuit d'induction. Remet le graphe en état propre avant de retourner.
"""
function find_circuit!(g::NeuroDSL.NeuroGraph, ns::Symbol, logits_sym::Symbol, window; max_sites::Int=6)
    tokens_clean, tokens_corrupt, j = window.tokens_clean, window.tokens_corrupt, window.j

    NeuroDSL.set!(g, :token_ids, tokens_clean; atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.set!(g, :pos_ids, collect(1:block_size); atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.invalidate_all!(g; namespace=ns)
    clean_output = copy(Array(NeuroDSL.demand!(g, logits_sym; namespace=ns)))
    clean_cache  = NeuroDSL.capture_activations(g, ns)   # demand! (pas demand_release!) -- capture_activations a besoin des valeurs intermédiaires

    NeuroDSL.set!(g, :token_ids, tokens_corrupt; atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.invalidate_all!(g; namespace=ns)
    corrupted_output = copy(Array(NeuroDSL.demand!(g, logits_sym; namespace=ns)))
    corrupted_cache  = NeuroDSL.capture_activations(g, ns)

    row_metric(out) = NeuroDSL.recovery_metric(Array(out)[j:j, :], clean_output[j:j, :], corrupted_output[j:j, :])

    candidates = sort(collect(filter(s -> occursin(r"_mha_ao_h\d+$", String(s)), keys(g.nodes[ns]))))

    NeuroDSL.set!(g, :token_ids, tokens_corrupt; atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.invalidate_all!(g; namespace=ns)
    NeuroDSL.demand!(g, logits_sym; namespace=ns)

    selected, trajectory = NeuroDSL.greedy_patch_search!(g, logits_sym, candidates, clean_cache, corrupted_cache,
                                                           clean_output, corrupted_output;
                                                           namespace=ns, max_sites=max_sites, metric=row_metric)
    remaining, pruned = if isempty(selected)
        (Symbol[], Symbol[])
    else
        NeuroDSL.backward_prune!(g, logits_sym, selected, clean_cache, corrupted_cache,
                                  clean_output, corrupted_output; namespace=ns, metric=row_metric)
    end

    # Vérification indépendante (ancre de sanité) : recovery du sous-ensemble
    # final recalculée depuis un état frais, via patch_nodes! direct.
    NeuroDSL.set!(g, :token_ids, tokens_corrupt; atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.invalidate_all!(g; namespace=ns)
    NeuroDSL.demand!(g, logits_sym; namespace=ns)
    isempty(remaining) || NeuroDSL.patch_nodes!(g, remaining, clean_cache; namespace=ns)
    out_check = NeuroDSL.demand!(g, logits_sym; namespace=ns)
    r_check = row_metric(out_check)

    # Remise en état propre (texte clean) pour la suite du notebook.
    NeuroDSL.set!(g, :token_ids, tokens_clean; atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.invalidate_all!(g; namespace=ns)
    NeuroDSL.demand!(g, logits_sym; namespace=ns)

    return (; candidates, selected, trajectory, remaining, pruned, r_check,
            n_candidates=length(candidates), clean_cache, corrupted_cache, clean_output, corrupted_output)
end

baseline_results = [find_circuit!(g, ns, logits_sym, w; max_sites=8) for w in frozen_windows]

println("Recherche de circuit -- BASELINE (4 couches, pas ", n_steps_A, ") :")
for (w, r) in zip(frozen_windows, baseline_results)
    println("  fenêtre '", w.name, "' (", w.variant, ", plafond=", round(w.ceiling,digits=3), ") : sites retenus après élagage = ", r.remaining,
            "  recovery vérifiée = ", round(r.r_check, digits=4), "  (", r.n_candidates, " candidats)")
end

Recherche de circuit -- BASELINE (4 couches, pas 10000) :
  fenêtre 'GRUMIO' (long_gap, plafond=0.84) : sites retenus après élagage = [:layer_4_mha_ao_h3, :layer_1_mha_ao_h2, :layer_3_mha_ao_h3, :layer_2_mha_ao_h2, :layer_3_mha_ao_h4, :layer_2_mha_ao_h4, :layer_1_mha_ao_h3, :layer_3_mha_ao_h2]  recovery vérifiée = 0.973  (16 candidats)
  fenêtre 'ANTONIO' (long_gap, plafond=0.977) : sites retenus après élagage = [:layer_1_mha_ao_h2, :layer_1_mha_ao_h4, :layer_1_mha_ao_h3, :layer_1_mha_ao_h1, :layer_4_mha_ao_h1, :layer_2_mha_ao_h2, :layer_2_mha_ao_h3, :layer_2_mha_ao_h1]  recovery vérifiée = 0.9975  (16 candidats)
  fenêtre 'PROSPERO' (long_gap, plafond=0.992) : sites retenus après élagage = [:layer_1_mha_ao_h1, :layer_1_mha_ao_h4, :layer_1_mha_ao_h2, :layer_2_mha_ao_h1, :layer_2_mha_ao_h2, :layer_1_mha_ao_h3, :layer_2_mha_ao_h3, :layer_2_mha_ao_h4]  recovery vérifiée = 1.0  (16 candidats)
  fenêtre 'GREMIO' (long_gap, plafond=0.996) : sites retenus après élagage = [:layer_1_mha_ao_h3, 

## 11. Étape 1 du redesign — balayage LARGE : `find_circuit!` sur les 55 candidats éligibles (pas seulement les 6 sélectionnés par plafond-couche-1)

**Pourquoi** : la sélection ci-dessus (§9) trie par `ceiling` -- la recovery
obtenue en patchant seulement les 4 têtes de couche 1. C'est un proxy PARTIEL :
un agent Fable a fait remarquer (session du 2026-07-11) que même la fenêtre au
plafond le plus bas (PETRUCHIO, ceiling=0.528) a ensuite atteint une recovery de
circuit **complet** (toutes couches) de 0.995 -- le plafond-couche-1 ne prédit
pas la saturation du circuit entier, seulement celle de la couche 1 isolée.
Avant de conclure que la tâche d'induction sur ce char-LM est saturée partout
(et donc de passer à la construction d'une tâche synthétique plus dure), on
vérifie ICI, à moindre coût, si un des 49 candidats éligibles jamais testés en
circuit complet (`eligible`, calculé au §9 mais utilisé seulement pour
`frozen_windows` jusqu'ici) révèle un vrai trou causal (recovery complète
&lt; 0.85) qui serait passé inaperçu. Aucun nouveau mécanisme -- réutilise
`find_circuit!` tel que défini au §10, juste sur un ensemble de fenêtres plus
large.

In [11]:
println("Balayage large : ", length(eligible), " candidats éligibles (vs ", length(frozen_windows), " sélectionnés jusqu'ici)\n")

t_wide_start = time()
wide_results = NamedTuple[]
for (i, w) in enumerate(eligible)
    r = find_circuit!(g, ns, logits_sym, w; max_sites=8)
    push!(wide_results, (; name=w.name, variant=w.variant, gap=w.gap, ceiling=w.ceiling,
                          r_check=r.r_check, n_selected=length(r.remaining)))
    if i % 10 == 0
        @printf("  ... %d/%d candidats traités (%.1f s écoulées)\n", i, length(eligible), time() - t_wide_start)
    end
end
t_wide_elapsed = time() - t_wide_start
@printf("\nBalayage large terminé : %d candidats en %.1f s (%.2f s/candidat moyen)\n",
        length(eligible), t_wide_elapsed, t_wide_elapsed / length(eligible))

sorted_wide = sort(wide_results, by = x -> x.r_check)
println("\nDistribution de la recovery COMPLÈTE (triée croissant, tous candidats éligibles) :")
for r in sorted_wide
    marker = r.r_check < 0.85 ? "  <-- SOUS 0.85" : ""
    @printf("  nom=%-14s variant=%-9s plafond(couche1)=%.3f  recovery_complète=%.4f  sites=%d%s\n",
            r.name, r.variant, r.ceiling, r.r_check, r.n_selected, marker)
end

below_085 = filter(x -> x.r_check < 0.85, wide_results)
println("\n", length(below_085), "/", length(wide_results),
        " candidats éligibles ont une recovery de circuit COMPLET sous 0.85.")
if isempty(below_085)
    println("=> Confirme le diagnostic de Fable : la saturation n'est pas un artefact de sous-échantillonnage")
    println("   (6 fenêtres) mais une propriété de cette tâche sur ce modèle à 4 couches. Passer à l'Étape 2")
    println("   (tâche synthétique à difficulté contrôlable, src/synthetic_circuits.jl) plutôt que de chercher")
    println("   davantage de fenêtres de texte réel.")
else
    println("=> Trou causal trouvé sans construire de nouvelle tâche -- ces fenêtres sont directement")
    println("   utilisables pour l'Étape 3 (protocole goulot-vs-témoin), à condition de vérifier qu'elles")
    println("   restent stables sur un second passage (bruit de mesure) avant de s'y engager.")
end

Balayage large : 8 candidats éligibles (vs 6 sélectionnés jusqu'ici)


Balayage large terminé : 8 candidats en 5.3 s (0.67 s/candidat moyen)

Distribution de la recovery COMPLÈTE (triée croissant, tous candidats éligibles) :
  nom=GRUMIO         variant=long_gap  plafond(couche1)=0.840  recovery_complète=0.9730  sites=8
  nom=GRUMIO         variant=long_gap  plafond(couche1)=0.996  recovery_complète=0.9974  sites=7
  nom=ANTONIO        variant=long_gap  plafond(couche1)=0.977  recovery_complète=0.9975  sites=8
  nom=GRUMIO         variant=adjacent  plafond(couche1)=0.916  recovery_complète=0.9981  sites=8
  nom=GREMIO         variant=long_gap  plafond(couche1)=0.996  recovery_complète=0.9985  sites=8
  nom=ARIEL          variant=long_gap  plafond(couche1)=0.998  recovery_complète=1.0000  sites=8
  nom=PROSPERO       variant=long_gap  plafond(couche1)=0.992  recovery_complète=1.0000  sites=8
  nom=BIANCA         variant=long_gap  plafond(couche1)=0.997  recovery_complète=1.0000  sites=8

In [12]:
using JSON

results_out = Dict(
    "n_eligible" => length(eligible),
    "n_below_085" => length(below_085),
    "wide_results" => wide_results,
    "elapsed_s" => t_wide_elapsed,
    "val_loss_phase_A" => result_A.val_history[end][2],
    "n_steps_A" => n_steps_A,
)
open(joinpath(@__DIR__, "real_llm_wide_screen_results.json"), "w") do io
    JSON.print(io, results_out)
end
println("Résultats écrits -> notebook/real_llm_wide_screen_results.json")

Résultats écrits -> notebook/real_llm_wide_screen_results.json
